# 第3回：画像分類とGrad-CAMによるAI判断の可視化

## 学習目標
- Grad-CAMを用いて、AIが判断時に画像のどの部分に注目しているかを可視化する

## Grad-CAMとは？
Grad-CAM（Gradient-weighted Class Activation Mapping; 勾配重み付けクラス活性化マッピング）は、
「深層学習モデルが判断を下す際に画像のどの部分に注目したか」を可視化する技術です。
- 赤色領域 → AIが強く注目した領域
- 青色領域 → AIが比較的注目しなかった領域


---
## はじめに

**操作方法**: `Shift + Enter` でセルを順番に実行してください。セッションが切断された場合は、上から再実行すれば復旧できます。

---

### この講義で挑戦すること

AIが「この画像は異常です」と判定したとき、あなたは信じられますか？

医療現場では「なぜそう判断したのか」を説明できないAIは信頼されません。ブラックボックスのままでは、医師も患者も納得できないからです。

この講義では、**Grad-CAM**という技術を使って、AIが画像の「どこを見て」判断しているのかを可視化します。AIの頭の中を覗いてみましょう。

---
## Step 1: 環境構築

In [ ]:
# 環境セットアップ
!pip install medmnist -q
import sys, os
!rm -rf /tmp/MedMNIST-Exercise
!git clone https://github.com/kshimoji8/MedMNIST-Exercise.git /tmp/MedMNIST-Exercise -q
sys.path.insert(0, '/tmp/MedMNIST-Exercise')
sys.modules.pop('exercise_logic', None)
import exercise_logic
exercise_logic.initialize_environment()
print("✓ セットアップが完了しました。")

# 胸部画像データのロード（ChestMNIST）
(x_train, y_train), (x_test, y_test), info = exercise_logic.load_and_preprocess(
    'chestmnist', 
    binary_classification=True
)

print(f"✓ 訓練データ: {x_train.shape}")
print(f"ラベル一覧: {info['label']}")


---
## データセットのサンプル画像

- まず、これから扱うデータセットがどのような画像なのか見てみましょう。
- ChestMNISTは、胸部X線画像の14クラス多クラス分類用データセットですが、本演習では「正常」か「異常」かの二値分類タスクとして再構成し利用します。
- ChestMNISTで扱う胸部疾患は、Atelectasis（無気肺）、Cardiomegaly（心肥大）、Consolidation（肺実質の浸潤影）、Edema（肺水腫・浮腫）、Effusion（胸水）、Emphysema（肺気腫）、Fibrosis（線維症）、Hernia（ヘルニア）、Infiltration（浸潤）、Mass（腫瘤）、Nodule（結節）、Pleural Thickening（胸膜肥厚）、Pneumonia（肺炎）、Pneumothorax（気胸）です。
- ChestMNISTは、すべての画像が統一されたフォーマット（28×28ピクセル）に前処理され、専門知識がなくても軽量で利用しやすいです。
- ChestMNISTは、研究・教育用ベンチマークとして設計され、臨床判断に直接用いることは想定されていません。
- ChestMNISTは、Creative Commons Attribution 4.0 International (CC BY 4.0) ライセンスの下で提供されており、適切な引用を行うことで再利用可能です。

In [ ]:
# データセットのサンプル画像を表示
exercise_logic.show_sample_images(x_train, n_samples=10)

---
## 二値分類とは

二値分類は、データを2つのカテゴリに分ける最もシンプルな分類タスクです。本演習では「正常」か「異常」かを判定します。

メールの「迷惑メールかどうか」、商品レビューの「ポジティブかネガティブか」なども二値分類です。Yes/Noで答えられる質問に対応しており、多クラス分類と比べて結果の解釈が直感的です。

---
## Step 2: データ準備とモデルトレーニング

ChestMNISTデータセット（胸部X線画像）を使用します。

ChestMNISTデータセットには14種類の疾患ラベルがありますが、本演習では**二値分類**に変換します
- **正常 (0)**: 異常なし
- **異常 (1)**: 何らかの異常が存在する

In [ ]:
# 二値分類モデルの構築と学習
model = exercise_logic.build_model(
    input_shape=(28, 28, 3), 
    num_classes=1  # Binary classification: 1 output unit with sigmoid
)

history = model.fit(x_train, y_train, epochs=5, validation_split=0.1, batch_size=128)

---
## Step 3:  Grad-CAMによる可視化

訓練済みモデルが診断のために画像のどの部分に焦点を当てているか見てみましょう

In [ ]:
# 最初の症例へのGrad-CAMの可視化
exercise_logic.show_gradcam(
    model, 
    x_test[0], 
    title_original="Chest X-ray",
    title_gradcam="AI Focus Region"
)

In [ ]:
# 複数の画像を比較する（最初の8症例）
exercise_logic.show_gradcam_comparison(
    model, 
    x_test[:8], 
    cols=4
)

---
### 練習：さまざまな画像を試す
`x_test[0]`の`0`を変更すると、異なる画像に対するGrad-CAMを表示できます。

In [ ]:
# 例：10症例目の画像を試してみてください
exercise_logic.show_gradcam(model, x_test[10])

### 練習：以下の点を検討してください

1. AIは胸部X線写真のどの領域に焦点を当てていますか？
2. これらの焦点領域は医学的に妥当であると考えますか？
3. Grad-CAMのような説明可能なAI技術は、臨床現場でどのように有用であると考えられますか？

---
## 設問演習

ここまでのGrad-CAM可視化結果を踏まえて、以下の設問に取り組んでください。

### 設問1：Grad-CAMの解釈と限界

先ほど表示されたGrad-CAMの結果を観察してください。

**(a)** AIの注目領域が肺野以外（画像の端、肩、横隔膜の下など）に集中している症例がありましたか？ もしあった場合、それは「AIが正しく学習している」証拠ですか、それとも問題の兆候ですか？理由とともに答えてください。

**(b)** 「Grad-CAMで赤く表示された領域 ＝ AIがその部分を"理解"している」と解釈してよいですか？ Grad-CAMが示しているものと、示していないものを区別して説明してください。

### 設問2：AIトリアージの運用設計

ある病院がこのモデルを胸部X線のトリアージ（緊急度の振り分け）に使うことを検討しています。放射線科医の読影前に、AIが「異常の疑いあり」と判定した画像を優先的に読影する運用です。

**(a)** この運用において、偽陰性（異常を正常と誤判定）と偽陽性（正常を異常と誤判定）のそれぞれがもたらすリスクを具体的に述べてください。

**(b)** Grad-CAMの結果を放射線科医に「AIの判断根拠」として提示することは有用ですか？ 有用である場合と、むしろ有害になりうる場合をそれぞれ考えてください。

---
## まとめ

- **二値分類**：画像を二つのカテゴリ（正常 vs 異常）に分類する手法
- **Grad-CAM**：CNNが「どこを見ているか」を可視化する技術
- **説明可能なAI（XAI）**：医療AIへの信頼構築に不可欠な技術

次回は「医療AIの評価指標」を学びます。ROC曲線・AUC・感度・特異度といった指標を理解し、適切な閾値設定ができるようになりましょう。

---
## 考察課題の回答例

以下は考察課題に対する回答の一例です。これが唯一の正解ではなく、議論の出発点として活用してください。

### 1. AIが焦点を当てる領域（胸部X線）

胸部X線では、可視化上は肺野・心陰影・横隔膜などの領域が注目されることが多い一方で、モデルが背景や画像周辺の情報に反応する場合もあり得る：

- 病変がある場合、理想的には病変周囲が強調されることが望ましい
- ただし、撮影条件やマーカー、周辺情報などに依存した"ショートカット特徴"を学習する可能性がある

### 2. 焦点領域の医学的妥当性の評価

注目領域の妥当性は慎重に評価する：

- 解剖学的に意味のある領域（肺、心臓など）に注目しているかを確認する
- 画像の端・背景・文字情報などが強く出る場合は、不適切な特徴学習の可能性を疑う
- 医師の注目領域との比較や、施設・機器を跨いだ外部検証で一貫性を確かめる

### 3. 説明可能なAI（例：Grad-CAM）の位置づけ

Grad-CAM等は有用だが、限界を理解したうえで使うべき：

- **デバッグ/品質管理**: 不適切な領域への依存を発見する助けになる
- **教育的利用**: 所見と対応する領域の議論材料になり得る
- **注意点**: 可視化は粗く、必ずしも「因果的根拠」を保証しないため、性能評価や外部検証とセットで解釈する
- **規制・品質の観点**: 説明可能性は万能要件ではなく、意図された使用とリスクに応じて、透明性・評価・監視などを総合的に整える

### 設問演習の回答例

### 設問1：Grad-CAMの解釈と限界

**(a)** 肺野以外への注目が見られた場合：

- これは**問題の兆候**である可能性が高い。AIが病変そのものではなく、撮影条件に由来するアーティファクト（画像の端のマーカー、患者の体格に由来する画像周辺の特徴など）を手がかりにしている可能性がある
- このような「ショートカット学習」は、訓練データ内では高い精度を出しつつも、撮影条件が異なるデータでは性能が大きく低下するリスクがある
- ただし、横隔膜付近への注目は、心拡大や胸水など一部の所見では医学的に妥当な場合もあるため、一概に問題とは言えず、所見との対応を確認する必要がある

**(b)** Grad-CAMが示すものと示さないもの：

- **示しているもの**: 最終的な分類判断に対して、最終畳み込み層の各チャネルがどの空間位置で勾配が大きいか（＝分類スコアへの感度が高い領域）の粗い近似
- **示していないもの**: AIがその領域を医学的に「理解」しているかどうか。因果関係（その領域が判断の原因であること）の保証。ピクセル単位の精密な根拠。また、他の手法（SHAP、Integrated Gradients等）とは異なる可視化結果が得られることもあり、単一の手法で「AIの判断根拠」を断定することには限界がある

### 設問2：AIトリアージの運用設計

**(a)** 偽陰性と偽陽性のリスク：

- **偽陰性（異常→正常と誤判定）のリスク**: 異常のある画像が「正常」と判定され、読影の優先度が下がる。結果として、緊急性の高い所見（気胸、大量胸水等）の発見が遅れ、治療開始の遅延につながり得る。トリアージの目的（緊急症例の早期発見）を根本的に損なう
- **偽陽性（正常→異常と誤判定）のリスク**: 正常画像が「異常疑い」として優先読影に回り、放射線科医の限られた時間が非効率に消費される。偽陽性が多すぎると、医師がAIの判定を信頼しなくなり、システム全体の運用が形骸化する恐れがある

**(b)** Grad-CAMの提示が有用な場合と有害な場合：

- **有用な場合**: 医師がGrad-CAMをあくまで参考情報（注目すべき領域のヒント）として使い、自身の読影を補完する場合。特に、見落としやすい小さな所見への注意喚起として機能し得る。また、AIの判定が不適切であることを医師が発見する手がかりにもなる
- **有害になりうる場合**: 医師がGrad-CAMの注目領域に引きずられ、それ以外の領域の観察が疎かになる場合（確証バイアスの増幅）。また、Grad-CAMが医学的に無関係な領域を強調している場合に、医師がそれを「AIの根拠」として過信し、誤った判断を補強してしまうリスクがある

---
## 発展的な学習（技術的詳細に興味のある方へ）

この講義では、技術的な詳細を `exercise_logic.py` に分離しています。
より深く学びたい方は、以下の関数のソースコードを参照してください。

### この講義で使用した主要関数

| 関数名 | 機能 | 技術的なポイント |
|--------|------|------------------|
| `initialize_environment()` | 環境セットアップ | Colab/Local判定、GPU設定 |
| `load_and_preprocess(binary_classification=True)` | 二値分類用データ変換 | マルチラベル→二値変換の仕組み |
| `show_sample_images()` | サンプル画像表示 | データセットの概観把握 |
| `build_model()` | CNNモデル構築 | last_conv_layerの命名とGrad-CAMとの連携 |
| `compute_gradcam()` | Grad-CAM計算 | 勾配計算、重み付き和、ReLU適用 |
| `show_gradcam()` | Grad-CAM可視化 | ヒートマップのオーバーレイ |
| `show_gradcam_comparison()` | 複数画像の比較 | バッチ処理による効率化 |

### ソースコードの参照方法

`exercise_logic.py` はGitHubリポジトリで公開しています：

https://github.com/kshimoji8/MedMNIST-Exercise/blob/main/exercise_logic.py

各関数には詳細な技術解説をdocstring（関数冒頭のコメント）として記載しています。